In [ ]:
import xarray as xr
import xclim.indices as xi

TASMAX_FILE = "tasmax.nc"
TASMIN_FILE = "tasmin.nc"
PR_FILE     = "pr.nc"

tasmax = xr.open_dataset(TASMAX_FILE, engine="h5netcdf")["tasmax"]
tasmin = xr.open_dataset(TASMIN_FILE, engine="h5netcdf")["tasmin"]
pr     = xr.open_dataset(PR_FILE,     engine="h5netcdf")["pr"]

# K → °C
tasmax = tasmax - 273.15
tasmax.attrs["units"] = "degC"

tasmin = tasmin - 273.15
tasmin.attrs["units"] = "degC"

# kg m-2 s-1 → mm/day
pr = pr * 86400
pr.attrs["units"] = "mm/day"

# =============================================================================
# COMPUTE INDICES 
# =============================================================================
TXx    = xi.tx_max(tasmax, freq="YS")
TNn    = xi.tn_min(tasmin, freq="YS")
RX1Day = xi.max_1day_precipitation_amount(pr, freq="YS")
RX5Day = xi.max_n_day_precipitation_amount(pr, window=5, freq="YS")

# SAVE
TXx.to_dataset(name="TXx").to_netcdf("TXx.nc")
TNn.to_dataset(name="TNn").to_netcdf("TNn.nc")
RX1Day.to_dataset(name="RX1Day").to_netcdf("RX1Day.nc")
RX5Day.to_dataset(name="RX5Day").to_netcdf("RX5Day.nc")

print("Done: TXx.nc | TNn.nc | RX1Day.nc | RX5Day.nc")

In [ ]:
import xarray as xr

# =============================================================================
# INPUT FILES — change these paths as needed
# =============================================================================
TASMAX_FILE = "tasmax.nc"
TASMIN_FILE = "tasmin.nc"
PR_FILE     = "pr.nc"

# =============================================================================
# OUTPUT FILES — output names are set automatically
# =============================================================================
OUT_TXX    = "TXx.nc"
OUT_TNN    = "TNn.nc"
OUT_RX1DAY = "RX1Day.nc"
OUT_RX5DAY = "RX5Day.nc"

# =============================================================================
# TXx — Annual maximum of daily maximum temperature
# =============================================================================
print("Computing TXx ...")
ds_tx = xr.open_dataset(TASMAX_FILE, engine="h5netcdf")
TXx = ds_tx["tasmax"].resample(time="YS").max(dim="time")
TXx.name = "TXx"
TXx.attrs.update({
    "long_name": "Annual Maximum Daily Maximum Temperature",
    "units":     ds_tx["tasmax"].attrs.get("units", "K"),
    "standard_name": "TXx",
})
TXx.to_dataset().to_netcdf(OUT_TXX)
ds_tx.close()
print(f"  Saved → {OUT_TXX}")

# =============================================================================
# TNn — Annual minimum of daily minimum temperature
# =============================================================================
print("Computing TNn ...")
ds_tn = xr.open_dataset(TASMIN_FILE, engine="h5netcdf")
TNn = ds_tn["tasmin"].resample(time="YS").min(dim="time")
TNn.name = "TNn"
TNn.attrs.update({
    "long_name": "Annual Minimum Daily Minimum Temperature",
    "units":     ds_tn["tasmin"].attrs.get("units", "K"),
    "standard_name": "TNn",
})
TNn.to_dataset().to_netcdf(OUT_TNN)
ds_tn.close()
print(f"  Saved → {OUT_TNN}")

# =============================================================================
# RX1Day — Annual maximum 1-day precipitation
# =============================================================================
print("Computing RX1Day ...")
ds_pr = xr.open_dataset(PR_FILE, engine="h5netcdf")
pr = ds_pr["pr"]

RX1Day = pr.resample(time="YS").max(dim="time")
RX1Day.name = "RX1Day"
RX1Day.attrs.update({
    "long_name": "Annual Maximum 1-Day Precipitation",
    "units":     pr.attrs.get("units", "kg m-2 s-1"),
    "standard_name": "RX1Day",
})
RX1Day.to_dataset().to_netcdf(OUT_RX1DAY)
print(f"  Saved → {OUT_RX1DAY}")

# =============================================================================
# RX5Day — Annual maximum consecutive 5-day precipitation
# =============================================================================
print("Computing RX5Day ...")

# Rolling 5-day sum along the time axis, then take annual max
pr_5day = pr.rolling(time=5, min_periods=5).sum()
RX5Day = pr_5day.resample(time="YS").max(dim="time")
RX5Day.name = "RX5Day"
RX5Day.attrs.update({
    "long_name": "Annual Maximum Consecutive 5-Day Precipitation",
    "units":     pr.attrs.get("units", "kg m-2 s-1"),
    "standard_name": "RX5Day",
})
RX5Day.to_dataset().to_netcdf(OUT_RX5DAY)
ds_pr.close()
print(f"  Saved → {OUT_RX5DAY}")

print("\nAll indices computed successfully.")